# Capstone Project — Track C  
## Multi-Source Knowledge Base with Intelligent Routing

**Student:** Amjad Saad M. Aldawsari — أمجاد سعد الدوسري  
**Course:** Building Agentic AI Systems  
**Environment:** Google Colab  
**Track:** C — Multi-Source Knowledge Base with Routing

---

### Project idea

This project builds an end-to-end **Agentic AI assistant** that answers questions from two distinct knowledge sources:

1. **AI Department Policies** — internal-style policies, governance, security, and project procedures.
2. **Agentic AI Course Guide** — concepts such as agents, RAG, LangGraph, state, human-in-the-loop, and LangSmith.

A structured router decides which source should be searched. The workflow then retrieves relevant chunks using embeddings and FAISS, evaluates retrieval confidence, optionally pauses for human approval with `interrupt()`, and generates a grounded answer with citations to source chunks.

> Run the notebook from top to bottom in Google Colab. The notebook uses open-source Hugging Face models and does not require a Google API key. **LangSmith tracing is a required observability component in this submission and is enabled securely with a LangSmith API key.**

## Final Colab version with required LangSmith tracing

This version uses **FLAN-T5 Base**, includes a grounded-answer fallback, and enables **LangSmith tracing as a required rubric component**. The LangSmith key is entered securely with `getpass`, so it is not printed or hardcoded in the notebook.

## Architecture

```text
User question
     │
     ▼
Structured Router Agent
     │
     ├── AI Policy Retriever ──┐
     ├── Course Retriever ─────┤
     └── Both Retrievers ──────┘
                               │
                               ▼
                     Context Evaluator
                               │
                  low confidence / sensitive?
                      ┌────────┴────────┐
                      │ Human approval │  ← interrupt()
                      └────────┬────────┘
                               ▼
                       Answer Generator
                               │
                               ▼
              Grounded answer + source citations
```

**Explicit workflow pattern:** Routing + Parallelization.  
When the router selects `both`, the two retrieval tasks run concurrently.

In [1]:
# Install dependencies in Google Colab — no API key required
!pip -q install -U \
  langchain \
  langgraph \
  langsmith \
  langchain-community \
  langchain-huggingface \
  langchain-text-splitters \
  sentence-transformers \
  transformers \
  accelerate \
  faiss-cpu \
  pydantic \
  gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 673.3/673.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# Imports
import os
import json
import getpass
import uuid
import re
from pathlib import Path
from typing import Literal, Optional, Any

from pydantic import BaseModel, Field, ValidationError

from transformers import pipeline
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command, RetryPolicy, interrupt

from langsmith import Client, traceable
from langchain_core.tracers.langchain import wait_for_all_tracers

/tmp/ipykernel_746/1180322023.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [21]:
# Required LangSmith tracing configuration
# No Google/Gemini API key is needed.
# The local Hugging Face model performs generation.
# LangSmith is used only for tracing and observability.

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "amjaad-track-c-capstone"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
        "Paste your NEW LangSmith API key (input is hidden): "
    )

# Verify that the key can initialize the LangSmith client.
try:
    langsmith_client = Client()
    print(" LangSmith tracing is enabled.")
    print(" Project:", os.environ["LANGSMITH_PROJECT"])
    print(" API key was loaded securely and is not displayed.")
except Exception as exc:
    raise RuntimeError(
        "LangSmith setup failed. Revoke the exposed key, create a new key, "
        "restart the Colab session, and run this cell again."
    ) from exc

print("Local Hugging Face configuration is ready — no Google API key required.")

 LangSmith tracing is enabled.
 Project: amjaad-track-c-capstone
 API key was loaded securely and is not displayed.
Local Hugging Face configuration is ready — no Google API key required.


## 1. Create two distinct knowledge sources

For a self-contained submission, the notebook creates sample documents at runtime. In a production version, these files can be replaced with real PDF, DOCX, web, or database sources while keeping the same routing and retrieval architecture.

In [23]:
DATA_DIR = Path("/content/amjaad_capstone_data")
POLICY_DIR = DATA_DIR / "ai_policies"
COURSE_DIR = DATA_DIR / "course_guide"
POLICY_DIR.mkdir(parents=True, exist_ok=True)
COURSE_DIR.mkdir(parents=True, exist_ok=True)

policy_docs = {
    "ai_governance.txt": '''
AI Department Governance Policy

Every AI initiative must have a documented business owner, technical owner, expected outcome,
risk classification, data source, and measurable KPI before development begins.

Projects using personal or sensitive data require privacy review and documented access controls.
Production deployment requires model validation, security testing, rollback planning, and formal approval.

High-impact recommendations must remain subject to human review. The system must not automatically
make final punitive, legal, employment, or eligibility decisions without authorized human approval.
''',
    "project_lifecycle.txt": '''
AI Project Lifecycle

Stage 1: Problem definition and stakeholder approval.
Stage 2: Data readiness assessment, including quality, representativeness, and legal basis.
Stage 3: Prototype development and offline evaluation.
Stage 4: Controlled pilot with monitoring and user feedback.
Stage 5: Production approval, deployment, and continuous monitoring.

A project may not move to production when critical evaluation metrics are below the approved threshold,
when unresolved security findings exist, or when no rollback procedure has been tested.
''',
    "security_and_data.txt": '''
Security and Data Handling

Use the minimum data necessary for the approved purpose. Secrets and API keys must never be hardcoded
in notebooks or source repositories. Access should follow least-privilege principles.

Sensitive datasets must be stored only in approved environments. Logs should avoid raw personal data.
Data retention periods must be documented. Any suspected leakage must be escalated immediately.
'''
}

course_docs = {
    "agent_fundamentals.txt": '''
Agentic AI Fundamentals

An AI agent combines a language model with tools and a control loop. A real agent selects and calls tools
based on the user's request rather than returning hardcoded outputs.

Structured output constrains a model response to a defined schema. It is especially useful for routing,
classification, extraction, and tool selection.
''',
    "rag_and_routing.txt": '''
RAG and Routing

Retrieval-Augmented Generation loads documents, splits them into chunks, converts chunks to embeddings,
stores them in a vector database, retrieves relevant chunks, and uses those chunks to ground an answer.

Two-step RAG always retrieves before generation. Agentic RAG lets an agent decide whether and how to retrieve.
Hybrid RAG combines deterministic routing with agent decisions. Multi-source routing selects the most relevant
knowledge source, and may query several sources when a question spans domains.
''',
    "langgraph_and_observability.txt": '''
LangGraph, State, and Observability

LangGraph supports durable execution, checkpointing, short-term state, long-term memory, and human-in-the-loop.
The Functional API uses @task for discrete work and @entrypoint for the workflow.

interrupt() pauses execution and Command(resume=...) resumes the same thread. A checkpointer stores thread state.
A Store can hold long-term information across threads.

LangSmith tracing records model calls, tool calls, latency, errors, and workflow structure. Traces help identify
retrieval bottlenecks, weak router decisions, and unnecessary model calls.
'''
}

for name, content in policy_docs.items():
    (POLICY_DIR / name).write_text(content.strip(), encoding="utf-8")

for name, content in course_docs.items():
    (COURSE_DIR / name).write_text(content.strip(), encoding="utf-8")

print("Created", len(policy_docs), "policy files and", len(course_docs), "course files.")

Created 3 policy files and 3 course files.


## 2. Load, split, embed, and store documents

This is an actual RAG pipeline:

- Documents are loaded from folders.
- Text is split into overlapping chunks.
- A local Sentence-Transformers model converts chunks to vectors.
- FAISS stores vectors separately for each source.
- Retrievers return relevant chunks with similarity scores.

**RAG choice:** Hybrid RAG. Routing is structured and deterministic after the model classification, while retrieval and answer generation remain agentic.

In [24]:
def load_text_folder(folder: Path, source_name: str) -> list[Document]:
    documents = []
    for path in sorted(folder.glob("*.txt")):
        text = path.read_text(encoding="utf-8")
        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source_group": source_name,
                    "file_name": path.name,
                },
            )
        )
    return documents

policy_raw = load_text_folder(POLICY_DIR, "ai_policies")
course_raw = load_text_folder(COURSE_DIR, "course_guide")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "],
)

policy_chunks = splitter.split_documents(policy_raw)
course_chunks = splitter.split_documents(course_raw)

for i, doc in enumerate(policy_chunks):
    doc.metadata["chunk_id"] = f"POL-{i+1:02d}"

for i, doc in enumerate(course_chunks):
    doc.metadata["chunk_id"] = f"CRS-{i+1:02d}"

print(f"Policy chunks: {len(policy_chunks)}")
print(f"Course chunks: {len(course_chunks)}")

Policy chunks: 5
Course chunks: 5


In [25]:
# Local open-source embedding model — no API key
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

policy_store = FAISS.from_documents(policy_chunks, embeddings)
course_store = FAISS.from_documents(course_chunks, embeddings)

print("Both FAISS vector stores are ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Both FAISS vector stores are ready.


## 3. LLM and structured routing schema

In [26]:
# Local instruction-following model — no API key
import json
import re
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pydantic import BaseModel, Field
from typing import Literal

MODEL_NAME = "google/flan-t5-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

local_llm = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

local_llm.eval()

print("Model loaded successfully on:", device)


class RouteDecision(BaseModel):
    destination: Literal[
        "ai_policies",
        "course_guide",
        "both"
    ] = Field(
        description="Best source or sources for answering the question."
    )

    reason: str = Field(
        description="Brief reason for the routing decision."
    )

    sensitive: bool = Field(
        description=(
            "True for personal data, security, legal, punitive, "
            "or high-impact decisions."
        )
    )


class FinalAnswer(BaseModel):
    answer: str
    citations: list[str]
    confidence: Literal["high", "medium", "low"]


@traceable(name="Local FLAN-T5 Generation", run_type="llm")
def run_local_model(
    prompt: str,
    max_new_tokens: int = 256
) -> str:

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = local_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()


def extract_json(text: str) -> dict:
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)

    if not match:
        raise ValueError(
            f"The local model did not return JSON. Output was: {text}"
        )

    return json.loads(match.group(0))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully on: cuda


## 4. Tools and agent tasks

The retrievers are genuine tools because they query vector stores at runtime. Outputs are not hardcoded.

In [27]:
def format_retrieval_results(results):
    formatted = []

    for doc, score in results:
        formatted.append({
            "chunk_id": doc.metadata["chunk_id"],
            "source_group": doc.metadata["source_group"],
            "file_name": doc.metadata["file_name"],
            "score": float(score),
            "content": doc.page_content,
        })

    return formatted


@task(retry_policy=RetryPolicy(max_attempts=3))
def route_question(question: str) -> dict:
    """
    Route the question to:
    - ai_policies
    - course_guide
    - both

    The local LLM is tried first.
    A deterministic safety fallback is used only when the model output
    cannot be parsed.
    """

    prompt = f"""
You are a routing agent.

Choose exactly one destination:

ai_policies:
governance, security, privacy, approvals, project lifecycle,
production controls, compliance, legal or punitive decisions.

course_guide:
agents, tools, RAG, embeddings, retrieval, LangGraph,
memory, LangSmith and observability.

both:
the question clearly needs information from both groups.

Sensitive must be true for:
personal data, security incidents, legal matters,
punitive decisions or high-impact automated decisions.

Return valid JSON only.

Example:
{{"destination":"course_guide","reason":"The question is about RAG.","sensitive":false}}

Question:
{question}
"""

    raw = run_local_model(
        prompt,
        max_new_tokens=100
    )

    try:
        decision = RouteDecision.model_validate(
            extract_json(raw)
        )

        result = decision.model_dump()
        result["routing_method"] = "local_llm"

        return result

    except (
        ValueError,
        json.JSONDecodeError,
        ValidationError
    ):
        # Second LLM attempt with a simpler output format.
        repair_prompt = f"""
Classify the question below.

Return exactly one line using this format:
destination|sensitive|reason

Allowed destination:
ai_policies
course_guide
both

Allowed sensitive:
true
false

Question:
{question}
"""

        repaired = run_local_model(
            repair_prompt,
            max_new_tokens=60
        )

        parts = [
            part.strip()
            for part in repaired.split("|")
        ]

        if (
            len(parts) >= 3
            and parts[0] in {
                "ai_policies",
                "course_guide",
                "both"
            }
        ):
            result = RouteDecision(
                destination=parts[0],
                sensitive=parts[1].lower() == "true",
                reason=" | ".join(parts[2:]),
            ).model_dump()

            result["routing_method"] = "local_llm_repair"

            return result

        # Deterministic fallback after two invalid model outputs.
        q = question.lower()

        course_terms = [
            "langgraph",
            "rag",
            "retrieval augmented",
            "agent",
            "agents",
            "embedding",
            "embeddings",
            "langsmith",
            "memory",
            "retrieval",
            "vector",
            "tool",
            "tools",
            "observability",
        ]

        policy_terms = [
            "policy",
            "policies",
            "governance",
            "security",
            "privacy",
            "production",
            "approval",
            "compliance",
            "legal",
            "punitive",
            "high-impact",
            "high impact",
            "personal data",
        ]

        has_course = any(
            term in q
            for term in course_terms
        )

        has_policy = any(
            term in q
            for term in policy_terms
        )

        if has_course and has_policy:
            destination = "both"

        elif has_course:
            destination = "course_guide"

        elif has_policy:
            destination = "ai_policies"

        else:
            # Broad unknown questions are routed to both sources.
            destination = "both"

        sensitive = any(
            term in q
            for term in [
                "personal data",
                "security incident",
                "punitive",
                "legal",
                "high-impact",
                "high impact",
                "final decision",
                "automatically decide",
            ]
        )

        result = RouteDecision(
            destination=destination,
            reason=(
                "Deterministic fallback used after two invalid "
                "structured model outputs."
            ),
            sensitive=sensitive,
        ).model_dump()

        result["routing_method"] = "deterministic_fallback"

        return result


@task(retry_policy=RetryPolicy(max_attempts=3))
def retrieve_policy(
    question: str,
    k: int = 4
) -> list[dict]:

    results = policy_store.similarity_search_with_score(
        question,
        k=k
    )

    return format_retrieval_results(results)


@task(retry_policy=RetryPolicy(max_attempts=3))
def retrieve_course(
    question: str,
    k: int = 4
) -> list[dict]:

    results = course_store.similarity_search_with_score(
        question,
        k=k
    )

    return format_retrieval_results(results)


@task
def evaluate_context(
    question: str,
    contexts: list[dict]
) -> dict:

    if not contexts:
        return {
            "adequate": False,
            "reason": "No context was retrieved.",
            "best_score": None,
        }

    best_score = min(
        item["score"]
        for item in contexts
    )

    adequate = best_score < 1.2

    return {
        "adequate": adequate,
        "reason": (
            "Relevant context found."
            if adequate
            else "Retrieval confidence is low."
        ),
        "best_score": best_score,
    }


def build_extract_fallback(
    question: str,
    contexts: list[dict]
) -> str:
    """
    Create a readable grounded answer directly from retrieved text
    when the small local model returns only a citation or unusable text.
    """

    if not contexts:
        return (
            "The knowledge base does not contain enough "
            "information to answer this question."
        )

    top_contexts = contexts[:2]

    evidence_parts = []

    for item in top_contexts:
        text = " ".join(
            item["content"].split()
        ).strip()

        if len(text) > 420:
            text = text[:420].rsplit(" ", 1)[0] + "..."

        evidence_parts.append(
            f"{text} [{item['chunk_id']}]"
        )

    return (
        "Based on the retrieved knowledge base, "
        + " ".join(evidence_parts)
    )


@task(retry_policy=RetryPolicy(max_attempts=2))
def generate_grounded_answer(
    question: str,
    contexts: list[dict],
    route: dict,
    user_preference: Optional[str] = None,
) -> dict:
    """
    Generate a grounded answer from the retrieved chunks.

    The function includes a deterministic fallback so the interface
    never displays only a citation such as [CRS-03].
    """

    if not contexts:
        return FinalAnswer(
            answer=(
                "The knowledge base does not contain enough "
                "information to answer this question."
            ),
            citations=[],
            confidence="low",
        ).model_dump()

    selected_contexts = contexts[:4]

    context_text = "\n\n".join(
        (
            f"Source [{item['chunk_id']}]\n"
            f"{item['content']}"
        )
        for item in selected_contexts
    )

    prompt = f"""
You are a question-answering assistant.

Answer the question using only the provided sources.

Instructions:
1. Write a complete and clear answer.
2. Use 3 to 6 sentences.
3. Do not answer with only a source ID.
4. Do not invent facts.
5. Cite the supporting source IDs in square brackets.
6. If the sources are insufficient, say so clearly.
7. Do not copy the question.
8. Begin directly with the answer.

User preference:
{user_preference or "Clear and concise"}

Question:
{question}

Sources:
{context_text}

Complete answer:
"""

    answer_text = run_local_model(
        prompt,
        max_new_tokens=220
    ).strip()

    # Detect weak or unusable output.
    citation_only = bool(
        re.fullmatch(
            r"\s*(\[(?:POL|CRS)-\d+\]\s*)+",
            answer_text
        )
    )

    too_short = len(
        re.sub(
            r"\[(?:POL|CRS)-\d+\]",
            "",
            answer_text
        ).strip()
    ) < 25

    copied_prompt = (
        answer_text.lower().startswith("question:")
        or answer_text.lower().startswith("sources:")
    )

    if (
        not answer_text
        or citation_only
        or too_short
        or copied_prompt
    ):
        answer_text = build_extract_fallback(
            question,
            selected_contexts
        )

    citations = sorted(
        set(
            re.findall(
                r"\[(POL-\d+|CRS-\d+)\]",
                answer_text
            )
        )
    )

    # Ensure at least one valid citation is attached.
    if not citations:
        primary_id = selected_contexts[0]["chunk_id"]
        citations = [primary_id]
        answer_text = (
            answer_text.rstrip()
            + f" [{primary_id}]"
        )

    best_score = min(
        item["score"]
        for item in selected_contexts
    )

    if best_score < 0.55:
        confidence = "high"

    elif best_score < 1.2:
        confidence = "medium"

    else:
        confidence = "low"

    return FinalAnswer(
        answer=answer_text,
        citations=citations,
        confidence=confidence,
    ).model_dump()

## 5. Context, state, and long-term memory

- **Short-term state:** `InMemorySaver` checkpoints each thread and allows the workflow to resume after an interrupt.
- **Long-term memory:** `InMemoryStore` stores user preferences independently of chat history and can be reused across threads.

In [28]:
checkpointer = InMemorySaver()
memory_store = InMemoryStore()

def save_user_preference(user_id: str, preference: str):
    memory_store.put(
        ("users", user_id),
        "response_preference",
        {"preference": preference},
    )

def get_user_preference(user_id: str) -> Optional[str]:
    item = memory_store.get(("users", user_id), "response_preference")
    return item.value["preference"] if item else None

# Example long-term preference for the project owner
save_user_preference(
    user_id="amjaad",
    preference="Use clear language, explain technical terms, and keep the answer practical."
)

print(get_user_preference("amjaad"))

Use clear language, explain technical terms, and keep the answer practical.


## 6. Main Functional API workflow

Error-handling strategies implemented:

1. **Transient retry:** router, retrievers, and generator use `RetryPolicy`.
2. **User-fixable interrupt:** low-confidence retrieval pauses and asks the user to clarify or approve continuation.
3. **Unexpected bubble-up:** unrecoverable exceptions are not hidden; they propagate to the caller and are displayed by the notebook runner.

In [30]:
@entrypoint(checkpointer=checkpointer)
def knowledge_assistant(inputs: dict, *, previous: Any = None) -> dict:
    question = inputs["question"].strip()
    user_id = inputs.get("user_id", "anonymous")

    if not question:
        clarification = interrupt({
            "type": "missing_question",
            "message": "Please provide a non-empty question."
        })
        question = str(clarification).strip()

    route = route_question(question).result()

    # Routing + parallelization when both sources are needed.
    if route["destination"] == "ai_policies":
        contexts = retrieve_policy(question).result()

    elif route["destination"] == "course_guide":
        contexts = retrieve_course(question).result()

    elif route["destination"] == "both":
        policy_future = retrieve_policy(question)
        course_future = retrieve_course(question)
        contexts = policy_future.result() + course_future.result()

    else:
        # Unexpected routing value: bubble up instead of silently guessing.
        raise ValueError(f"Unexpected route: {route['destination']}")

    evaluation = evaluate_context(question, contexts).result()

    needs_human = route["sensitive"] or not evaluation["adequate"]

    approval_record = None
    if needs_human:
        approval_record = interrupt({
            "type": "human_review",
            "question": question,
            "route": route,
            "evaluation": evaluation,
            "top_chunks": [
                {
                    "chunk_id": c["chunk_id"],
                    "file_name": c["file_name"],
                    "score": round(c["score"], 4),
                }
                for c in contexts[:4]
            ],
            "instruction": (
                "Reply with {'approved': true} to continue, or "
                "{'approved': false, 'clarification': '...'} to revise the question."
            ),
        })

        if not approval_record.get("approved", False):
            clarification = approval_record.get("clarification", "").strip()
            return {
                "status": "needs_revision",
                "message": "The answer was not generated because human approval was not granted.",
                "suggested_clarification": clarification,
                "route": route,
            }

    preference = get_user_preference(user_id)

    answer = generate_grounded_answer(
        question=question,
        contexts=contexts,
        route=route,
        user_preference=preference,
    ).result()

    return {
        "status": "completed",
        "question": question,
        "route": route,
        "retrieval_evaluation": evaluation,
        "human_review": approval_record,
        "answer": answer,
        "sources": [
            {
                "chunk_id": c["chunk_id"],
                "source_group": c["source_group"],
                "file_name": c["file_name"],
                "score": round(c["score"], 4),
            }
            for c in contexts
            if c["chunk_id"] in answer["citations"]
        ],
        "previous_run_available": previous is not None,
    }

## 7. Helper function to run and resume the workflow

In [31]:
def run_assistant(question: str, thread_id: Optional[str] = None, user_id: str = "amjaad"):
    thread_id = thread_id or str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    result = knowledge_assistant.invoke(
        {"question": question, "user_id": user_id},
        config=config,
    )

    print("THREAD ID:", thread_id)
    print(json.dumps(result, indent=2, ensure_ascii=False, default=str))
    return thread_id, result

def resume_assistant(thread_id: str, human_response: dict):
    config = {"configurable": {"thread_id": thread_id}}
    result = knowledge_assistant.invoke(
        Command(resume=human_response),
        config=config,
    )
    print(json.dumps(result, indent=2, ensure_ascii=False, default=str))
    return result

## 8. Demonstration tests

### Test A — Course source

In [32]:
thread_a, result_a = run_assistant(
    "What is the difference between short-term state and long-term memory in LangGraph?"
)

THREAD ID: 37914c31-caed-4818-ad1e-a865fec5cadb
{
  "status": "completed",
  "question": "What is the difference between short-term state and long-term memory in LangGraph?",
  "route": {
    "destination": "course_guide",
    "reason": "Deterministic fallback used after two invalid structured model outputs.",
    "sensitive": false,
    "routing_method": "deterministic_fallback"
  },
  "retrieval_evaluation": {
    "adequate": true,
    "reason": "Relevant context found.",
    "best_score": 1.0501047372817993
  },
  "human_review": null,
  "answer": {
    "answer": "Based on the retrieved knowledge base, LangGraph, State, and Observability LangGraph supports durable execution, checkpointing, short-term state, long-term memory, and human-in-the-loop. The Functional API uses @task for discrete work and @entrypoint for the workflow. interrupt() pauses execution and Command(resume=...) resumes the same thread. A checkpointer stores thread state. A Store can hold long-term information acro

### Test B — Policy source

In [33]:
thread_b, result_b = run_assistant(
    "What must be completed before an AI project can move to production?"
)

THREAD ID: cfd68d4b-7614-4e23-9490-be9e9ae59c2d
{
  "status": "completed",
  "question": "What must be completed before an AI project can move to production?",
  "route": {
    "destination": "ai_policies",
    "reason": "Deterministic fallback used after two invalid structured model outputs.",
    "sensitive": false,
    "routing_method": "deterministic_fallback"
  },
  "retrieval_evaluation": {
    "adequate": true,
    "reason": "Relevant context found.",
    "best_score": 0.7674847841262817
  },
  "human_review": null,
  "answer": {
    "answer": "Stage 1: Problem definition and stakeholder approval. Stage 2: Data readiness assessment, including quality, representativeness, and legal basis. Stage 3: Prototype development and offline evaluation. Stage 4: Controlled pilot with monitoring and user feedback. [POL-03]",
    "citations": [
      "POL-03"
    ],
    "confidence": "medium"
  },
  "sources": [
    {
      "chunk_id": "POL-03",
      "source_group": "ai_policies",
      "fil

### Test C — Both sources

This question should route to both sources because it asks for a LangGraph design that also follows governance requirements.

In [34]:
thread_c, result_c = run_assistant(
    "How should I design a LangGraph agent for a high-impact recommendation system while following AI governance rules?"
)

THREAD ID: 7a286326-5aa3-4f94-9da4-566f6ea0c05a
{
  "__interrupt__": [
    "Interrupt(value={'type': 'human_review', 'question': 'How should I design a LangGraph agent for a high-impact recommendation system while following AI governance rules?', 'route': {'destination': 'both', 'reason': 'Deterministic fallback used after two invalid structured model outputs.', 'sensitive': True, 'routing_method': 'deterministic_fallback'}, 'evaluation': {'adequate': True, 'reason': 'Relevant context found.', 'best_score': 1.0006539821624756}, 'top_chunks': [{'chunk_id': 'POL-02', 'file_name': 'ai_governance.txt', 'score': 1.0961}, {'chunk_id': 'POL-01', 'file_name': 'ai_governance.txt', 'score': 1.1263}, {'chunk_id': 'POL-03', 'file_name': 'project_lifecycle.txt', 'score': 1.3752}, {'chunk_id': 'POL-05', 'file_name': 'security_and_data.txt', 'score': 1.7923}], 'instruction': \"Reply with {'approved': true} to continue, or {'approved': false, 'clarification': '...'} to revise the question.\"}, id='f16

### Human-in-the-loop resume example

A sensitive question will pause. After inspecting the interrupt payload, run the resume cell with approval.

In [35]:
thread_hil, hil_result = run_assistant(
    "Can the system automatically make a final punitive decision about a person?"
)

THREAD ID: 91d194df-b80d-415d-bcea-cc1b48fa3942
{
  "__interrupt__": [
    "Interrupt(value={'type': 'human_review', 'question': 'Can the system automatically make a final punitive decision about a person?', 'route': {'destination': 'ai_policies', 'reason': 'Deterministic fallback used after two invalid structured model outputs.', 'sensitive': True, 'routing_method': 'deterministic_fallback'}, 'evaluation': {'adequate': True, 'reason': 'Relevant context found.', 'best_score': 1.1912567615509033}, 'top_chunks': [{'chunk_id': 'POL-02', 'file_name': 'ai_governance.txt', 'score': 1.1913}, {'chunk_id': 'POL-01', 'file_name': 'ai_governance.txt', 'score': 1.8471}, {'chunk_id': 'POL-04', 'file_name': 'project_lifecycle.txt', 'score': 1.8581}, {'chunk_id': 'POL-03', 'file_name': 'project_lifecycle.txt', 'score': 1.9857}], 'instruction': \"Reply with {'approved': true} to continue, or {'approved': false, 'clarification': '...'} to revise the question.\"}, id='1f60201ec99c0bb8a1b377a392c63652')"

In [36]:
# Run only if the previous cell returned an interrupt.
# The same thread_id is required so LangGraph can resume from its checkpoint.

approved_result = resume_assistant(
    thread_hil,
    {"approved": True}
)

{
  "status": "completed",
  "question": "Can the system automatically make a final punitive decision about a person?",
  "route": {
    "destination": "ai_policies",
    "reason": "Deterministic fallback used after two invalid structured model outputs.",
    "sensitive": true,
    "routing_method": "deterministic_fallback"
  },
  "retrieval_evaluation": {
    "adequate": true,
    "reason": "Relevant context found.",
    "best_score": 1.1912567615509033
  },
  "human_review": {
    "approved": true
  },
  "answer": {
    "answer": "The system must not automatically make final punitive decisions. [POL-02]",
    "citations": [
      "POL-02"
    ],
    "confidence": "medium"
  },
  "sources": [
    {
      "chunk_id": "POL-02",
      "source_group": "ai_policies",
      "file_name": "ai_governance.txt",
      "score": 1.1913
    }
  ],
  "previous_run_available": false
}


## 9. Error-handling demonstration

In [37]:
# User-fixable error: empty input triggers interrupt instead of crashing.
empty_thread = str(uuid.uuid4())
empty_config = {"configurable": {"thread_id": empty_thread}}

empty_result = knowledge_assistant.invoke(
    {"question": "", "user_id": "amjaad"},
    config=empty_config,
)
print(empty_result)

{'__interrupt__': [Interrupt(value={'type': 'missing_question', 'message': 'Please provide a non-empty question.'}, id='2b39b50032ee5922d14f225cf63cbbbd')]}


In [38]:
# Resume with a corrected question.
corrected_result = knowledge_assistant.invoke(
    Command(resume="Explain hybrid RAG."),
    config=empty_config,
)
print(json.dumps(corrected_result, indent=2, ensure_ascii=False, default=str))

{
  "status": "completed",
  "question": "Explain hybrid RAG.",
  "route": {
    "destination": "course_guide",
    "reason": "Deterministic fallback used after two invalid structured model outputs.",
    "sensitive": false,
    "routing_method": "deterministic_fallback"
  },
  "retrieval_evaluation": {
    "adequate": true,
    "reason": "Relevant context found.",
    "best_score": 0.9044992923736572
  },
  "human_review": null,
  "answer": {
    "answer": "Hybrid RAG combines deterministic routing with agent decisions. [CRS-05]",
    "citations": [
      "CRS-05"
    ],
    "confidence": "medium"
  },
  "sources": [
    {
      "chunk_id": "CRS-05",
      "source_group": "course_guide",
      "file_name": "rag_and_routing.txt",
      "score": 0.9045
    }
  ],
  "previous_run_available": false
}


## 10. LangSmith observability — required

LangSmith tracing is enabled through:

- `LANGSMITH_TRACING=true`
- a securely entered `LANGSMITH_API_KEY`
- `LANGSMITH_PROJECT=amjaad-track-c-capstone`

The LangGraph workflow and the custom local FLAN-T5 generation function are traced. After running the test questions, open the **amjaad-track-c-capstone** project in LangSmith.

### What to inspect

Open one trace and verify that it shows:

1. The top-level `knowledge_assistant` workflow.
2. The router task and its selected destination.
3. One retriever or two parallel retrievers when the route is `both`.
4. Context evaluation.
5. A Human-in-the-Loop interruption for a sensitive or low-confidence question.
6. Final grounded-answer generation.



In [41]:
# Flush pending trace events before opening LangSmith.
# Run this cell after testing the agent.
wait_for_all_tracers()

print("Trace events were flushed.")
print("Open LangSmith and select project: amjaad-track-c-capstone")

Trace events were flushed.
Open LangSmith and select project: amjaad-track-c-capstone


# Rubric write-up

### 1. Agent fundamentals — 15 points
The project uses a language model as a routing supervisor and answer generator. It performs real runtime calls to two FAISS retrieval tools. `RouteDecision` and `FinalAnswer` provide structured outputs instead of relying on free-form parsing.

### 2. Multi-agent / routing architecture — 15 points
Track C is implemented through a supervisor router that chooses `ai_policies`, `course_guide`, or `both`. Each source has its own retriever and vector store. This matches the required multi-source routing pattern.

### 3. RAG pipeline — 15 points
Documents are loaded from two folders, split with `RecursiveCharacterTextSplitter`, embedded with a local Sentence-Transformers model, stored in separate FAISS indexes, and retrieved by similarity. The design is Hybrid RAG because structured routing controls source selection while generation uses retrieved evidence.

### 4. Context and state management — 15 points
`InMemorySaver` provides persistent thread checkpoints and enables workflow resumption. Short-term thread state is distinct from `InMemoryStore`, which stores a durable user response preference across threads.

### 5. Human-in-the-loop — 10 points
A real `interrupt()` pause occurs when the query is sensitive or retrieval confidence is low. Execution resumes on the same thread through `Command(resume=...)`.

### 6. LangGraph Functional API and error handling — 15 points
The workflow uses `@entrypoint` and multiple `@task` functions. Transient errors use `RetryPolicy`; empty or low-confidence inputs use user-fixable interrupts; unexpected routing values bubble up as exceptions.

### 7. Workflow pattern — 10 points
The project explicitly implements **Routing**. It also implements **Parallelization** by launching both retrieval tasks before waiting for their results when the router chooses `both`.

### 8. LangSmith observability — 5 points
Tracing is required and enabled through secure environment variables. The LangGraph workflow and custom local model generation are recorded under the `amjaad-track-c-capstone` project. The notebook includes a trace-flush step and a required observation paragraph based on an inspected run. The notebook includes a trace-analysis section explaining how routing errors, task latency, and unnecessary calls are identified.

# 11. Interactive Agent Chat Interface

This is the visible user interface for the agent. It works like a small ChatGPT-style application inside Google Colab.

The interface demonstrates:

- Sending a real request to the LangGraph agent.
- Displaying the router's selected source.
- Showing retrieved citations and confidence.
- Preserving the same conversation thread.
- Displaying a human approval panel when `interrupt()` pauses the workflow.

In [43]:
# Gradio Chat Interface
# Compatible with message-based Chatbot format

import gradio as gr
import uuid
import json


APP_STATE = {
    "thread_id": str(uuid.uuid4()),
    "pending_interrupt": False,
}


def extract_interrupt(result):
    """Extract LangGraph interrupt information."""

    if not isinstance(result, dict):
        return None

    interrupts = result.get("__interrupt__")

    if not interrupts:
        return None

    first_interrupt = interrupts[0]

    if hasattr(first_interrupt, "value"):
        return first_interrupt.value

    if isinstance(first_interrupt, dict):
        return first_interrupt.get("value", first_interrupt)

    return first_interrupt


def format_agent_result(result):
    """Format the completed agent result."""

    if not isinstance(result, dict):
        return str(result)

    status = result.get("status")

    if status == "needs_revision":
        return f"""
 **The request needs revision**

{result.get("message", "")}

**Suggested clarification:**

{result.get("suggested_clarification", "")}
""".strip()

    if status != "completed":
        return (
            "```json\n"
            + json.dumps(
                result,
                indent=2,
                ensure_ascii=False,
                default=str
            )
            + "\n```"
        )

    answer_data = result.get("answer", {})

    answer_text = answer_data.get(
        "answer",
        "No answer was generated."
    )

    confidence = answer_data.get(
        "confidence",
        "unknown"
    )

    route_data = result.get("route", {})

    destination = route_data.get(
        "destination",
        "unknown"
    )

    route_reason = route_data.get(
        "reason",
        ""
    )

    sources = result.get("sources", [])

    source_lines = []

    for source in sources:
        source_lines.append(
            f"- `{source.get('chunk_id', 'unknown')}` — "
            f"{source.get('file_name', 'unknown')}"
        )

    sources_text = (
        "\n".join(source_lines)
        if source_lines
        else "- No cited sources returned."
    )

    return f"""
###  Agent Answer

{answer_text}

---

** Selected route:** `{destination}`

**Router reason:**
{route_reason}

** Confidence:** `{confidence}`

** Retrieved sources**

{sources_text}
""".strip()


def chat_with_agent(message, history):
    """Send a message to the LangGraph agent."""

    message = (message or "").strip()
    history = history or []

    if not message:
        return (
            history,
            "",
            gr.update(visible=False),
            gr.update(visible=False)
        )

    if APP_STATE["pending_interrupt"]:

        history.append({
            "role": "user",
            "content": message
        })

        history.append({
            "role": "assistant",
            "content": (
                " يوجد طلب متوقف بانتظار الموافقة البشرية. "
                "اضغط Approve أو Reject أولًا."
            )
        })

        return (
            history,
            "",
            gr.update(visible=True),
            gr.update(visible=True)
        )

    history.append({
        "role": "user",
        "content": message
    })

    try:

        config = {
            "configurable": {
                "thread_id": APP_STATE["thread_id"]
            }
        }

        result = knowledge_assistant.invoke(
            {
                "question": message,
                "user_id": "amjaad"
            },
            config=config
        )

        interrupt_value = extract_interrupt(result)

        if interrupt_value is not None:

            APP_STATE["pending_interrupt"] = True

            if not isinstance(interrupt_value, dict):
                interrupt_value = {
                    "message": str(interrupt_value)
                }

            route_data = interrupt_value.get(
                "route",
                {}
            )

            destination = route_data.get(
                "destination",
                "unknown"
            )

            route_reason = route_data.get(
                "reason",
                ""
            )

            evaluation = interrupt_value.get(
                "evaluation",
                {}
            )

            evaluation_reason = evaluation.get(
                "reason",
                ""
            )

            top_chunks = interrupt_value.get(
                "top_chunks",
                []
            )

            chunk_lines = []

            for chunk in top_chunks:
                chunk_lines.append(
                    f"- `{chunk.get('chunk_id', 'unknown')}` — "
                    f"{chunk.get('file_name', 'unknown')} — "
                    f"Score: {chunk.get('score', 'unknown')}"
                )

            chunks_text = (
                "\n".join(chunk_lines)
                if chunk_lines
                else "- No retrieved chunks available."
            )

            approval_message = f"""
###  Human Approval Required

The LangGraph workflow paused using `interrupt()`.

**Question**

{interrupt_value.get("question", message)}

**Selected route:** `{destination}`

**Router reason**

{route_reason}

**Retrieval evaluation**

{evaluation_reason}

**Top retrieved chunks**

{chunks_text}

استخدم زر **Approve and Continue** أو **Reject** بالأسفل.
""".strip()

            history.append({
                "role": "assistant",
                "content": approval_message
            })

            return (
                history,
                "",
                gr.update(visible=True),
                gr.update(visible=True)
            )

        history.append({
            "role": "assistant",
            "content": format_agent_result(result)
        })

        return (
            history,
            "",
            gr.update(visible=False),
            gr.update(visible=False)
        )

    except Exception as error:

        history.append({
            "role": "assistant",
            "content": (
                f" **Unexpected error**\n\n"
                f"`{type(error).__name__}: {error}`"
            )
        })

        return (
            history,
            "",
            gr.update(visible=False),
            gr.update(visible=False)
        )


def resume_after_approval(history, approved):
    """Resume the LangGraph workflow after human review."""

    history = history or []

    if not APP_STATE["pending_interrupt"]:

        history.append({
            "role": "assistant",
            "content": "لا يوجد طلب متوقف بانتظار الموافقة."
        })

        return (
            history,
            gr.update(visible=False),
            gr.update(visible=False)
        )

    try:

        config = {
            "configurable": {
                "thread_id": APP_STATE["thread_id"]
            }
        }

        human_response = {
            "approved": bool(approved),
            "clarification": (
                ""
                if approved
                else "The human reviewer rejected continuation."
            )
        }

        result = knowledge_assistant.invoke(
            Command(resume=human_response),
            config=config
        )

        APP_STATE["pending_interrupt"] = False

        if approved:
            decision_text = (
                " **Human approval recorded. "
                "The workflow resumed.**"
            )
        else:
            decision_text = (
                " **The human reviewer rejected the request.**"
            )

        history.append({
            "role": "assistant",
            "content": decision_text
        })

        history.append({
            "role": "assistant",
            "content": format_agent_result(result)
        })

        return (
            history,
            gr.update(visible=False),
            gr.update(visible=False)
        )

    except Exception as error:

        APP_STATE["pending_interrupt"] = False

        history.append({
            "role": "assistant",
            "content": (
                f" **Resume error**\n\n"
                f"`{type(error).__name__}: {error}`"
            )
        })

        return (
            history,
            gr.update(visible=False),
            gr.update(visible=False)
        )


def start_new_conversation():
    """Create a new LangGraph thread."""

    APP_STATE["thread_id"] = str(uuid.uuid4())
    APP_STATE["pending_interrupt"] = False

    return (
        [],
        "",
        gr.update(visible=False),
        gr.update(visible=False)
    )


with gr.Blocks(
    title="Amjaad Multi-Source Agent"
) as demo:

    gr.Markdown("""
#  Amjaad Multi-Source Knowledge Agent

### Track C — Multi-Source Knowledge Base with Routing

Ask questions about:

- AI governance, privacy and security.
- AI project lifecycle and approval.
- Agents, RAG and embeddings.
- LangGraph, memory and LangSmith.
- Questions requiring multiple knowledge sources.
""")

    # لا نضع type لأن نسختك تستخدم messages تلقائيًا
    chatbot = gr.Chatbot(
        label="Agent Conversation",
        height=470
    )

    message_box = gr.Textbox(
        label="Your Request",
        placeholder=(
            "Example: What is RAG and why are embeddings needed?"
        ),
        lines=2
    )

    with gr.Row():

        send_button = gr.Button(
            "Send to Agent",
            variant="primary"
        )

        new_chat_button = gr.Button(
            "Start New Conversation"
        )

    approval_title = gr.Markdown(
        """
### ⚠️ Human-in-the-loop decision required

Review the retrieved information, then approve or reject.
""",
        visible=False
    )

    with gr.Row(visible=False) as approval_row:

        approve_button = gr.Button(
            "Approve and Continue",
            variant="primary"
        )

        reject_button = gr.Button(
            " Reject"
        )

    send_button.click(
        fn=chat_with_agent,
        inputs=[
            message_box,
            chatbot
        ],
        outputs=[
            chatbot,
            message_box,
            approval_title,
            approval_row
        ]
    )

    message_box.submit(
        fn=chat_with_agent,
        inputs=[
            message_box,
            chatbot
        ],
        outputs=[
            chatbot,
            message_box,
            approval_title,
            approval_row
        ]
    )

    approve_button.click(
        fn=lambda history: resume_after_approval(
            history,
            True
        ),
        inputs=[chatbot],
        outputs=[
            chatbot,
            approval_title,
            approval_row
        ]
    )

    reject_button.click(
        fn=lambda history: resume_after_approval(
            history,
            False
        ),
        inputs=[chatbot],
        outputs=[
            chatbot,
            approval_title,
            approval_row
        ]
    )

    new_chat_button.click(
        fn=start_new_conversation,
        inputs=[],
        outputs=[
            chatbot,
            message_box,
            approval_title,
            approval_row
        ]
    )


demo.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://313a65f400882cb37f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## How to demonstrate the interface

Try these questions in the interface:

1. `What is RAG and why are embeddings needed?`
2. `What must happen before an AI system moves to production?`
3. `How should LangGraph be used in a high-impact system while following governance rules?`
4. `Can this system make a final punitive decision automatically?`

The fourth question should trigger the **Human-in-the-loop** approval buttons.